# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [2]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

import sys
sys.path.append('../05_src/')

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [3]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "documents/ai_report_2025.pdf"
loader = PyPDFLoader(pdf_path)
docs = loader.load()

document_text = ""
for page in docs:
     document_text += page.page_content + "\n"

print(f"Pages: {len(docs)} | Characters: {len(document_text)}")
print(document_text[:600])

C:\Users\roman\AppData\Local\Temp\ipykernel_205260\1265036117.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Pages: 26 | Characters: 53851
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI initiatives, structured 
interviews with representatives from 52 organizations, and survey responses f


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
#Schema plan:
 #   Split into two models:
    #      SummaryFields to hold five fields the model produces,
    #      ArticleSummary adds InputTokens/OutputTokens separately. Also model would have to get tokens stats from telemetry read after its called.

 # Model used: gpt-4o-mini.

In [ ]:
from pydantic import BaseModel, Field

class SummaryFields(BaseModel):
    Author: str = Field(description="The author(s) of the article, taken from the document. Do not fabricate.")
    Title: str = Field(description="The article's title, exactly as it appears in the document.")
    Relevance: str = Field(description="At most one paragraph on why this article matters for an AI professional's development")
    Summary: str = Field(description="A faithful summary of the article, ~1000 tokens max, written entirely in the requested tone")
    Tone: str = Field(description="The name of the tone used to write the Summary, e.g. 'Victorian English'.")

class ArticleSummary(SummaryFields):
    InputTokens: int
    OutputTokens: int


In [5]:
TONE = "Victorian English"
developer_prompt = """You are an expert analyst who writes structured briefs about articles for AI professionals.
Rules:
- Write the Summary entirely in {tone}: use the period-appropriate vocabulary, register, and
  sentence construction of that style, while remaining strictly faithful to the source. Invent nothing.
- Keep the Summary concise: no more than ~1000 tokens.
- Relevance must be at most one paragraph, focused on professional development for an AI practitioner.
- Take Author and Title directly from the document. If genuinely absent, say so; never fabricate.
- The Tone field must name the tone you actually used.
""".format(tone=TONE)

user_prompt = """
Produce the structured brief for the following article.
<article>
{document}
</article>
""".format(document=document_text)

In [6]:
print(developer_prompt)

You are an expert analyst who writes structured briefs about articles for AI professionals.
Rules:
- Write the Summary entirely in Victorian English: use the period-appropriate vocabulary, register, and
  sentence construction of that style, while remaining strictly faithful to the source. Invent nothing.
- Keep the Summary concise: no more than ~1000 tokens.
- Relevance must be at most one paragraph, focused on professional development for an AI practitioner.
- Take Author and Title directly from the document. If genuinely absent, say so; never fabricate.
- The Tone field must name the tone you actually used.



In [8]:
import os
from utils.clients import get_client

MODEL = os.getenv("MODEL", "gpt-4o-mini")
client = get_client()
response = client.responses.parse(model=MODEL, instructions=developer_prompt, input=[{"role":"user","content": user_prompt}], text_format=SummaryFields, temperature=0.7,)
parsed = response.output_parsed
article_summary = ArticleSummary(**parsed.model_dump(),InputTokens=response.usage.input_tokens,OutputTokens=response.usage.output_tokens,)

article_summary

ArticleSummary(Author='MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari', Title='The GenAI Divide: State of AI in Business 2025', Relevance='This article provides essential insights into the current state of Generative AI adoption in enterprises, highlighting the stark contrasts between high adoption rates and low transformative impact. For AI professionals, understanding these dynamics is crucial for effective implementation and strategy formulation in their organizations, as it illuminates common pitfalls and successful approaches in the rapidly evolving landscape of AI technology.', Summary="In the year of our Lord 2025, a comprehensive report emerges from the esteemed Project NANDA, elucidating the state of Generative AI (GenAI) within the enterprise sphere. Despite a staggering investment ranging from $30 to $40 billion, a lamentable 95% of organizations have reaped naught in terms of return on investment, illustrating a profound dichotomy henceforth term

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:

#I used course GATEWAY to write and judge, but for better results its beneficial to run justdge on a separate model. 

from deepeval.models import GPTModel
import os

USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

if USE_GATEWAY:
    judge_model = GPTModel(model=MODEL, temperature=1, api_key='any value',default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
                            base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',)
else:
    judge_model = GPTModel(model=MODEL, temperature=1)

judge_model

In [ ]:
#since tonality metric failed at reading Victorian voice and scored 0.01, I had to rewrite it to adjust to the modern words and judge prose instead.

from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase

assessment_questions = [
    "Does the summary state that around 95% of organizations saw no return on their GenAI investment?",
    "Does the summary explain the concept of the 'GenAI Divide'?",
    "Does the summary mention that external partnerships succeed at roughly twice the rate of internal builds?",
    "Does the summary describe the 'shadow AI economy' where employees use personal AI tools?",
    "Does the summary attribute the divide to adoption/learning approaches rather than model quality or regulation?",
]

summarization_metric = SummarizationMetric(threshold=0.5, model=judge_model, assessment_questions=assessment_questions, include_reason=True,)

test_case = LLMTestCase(input=document_text, actual_output=article_summary.Summary,)

summarization_metric.measure(test_case)

print("Score:", summarization_metric.score)
print("Reason:", summarization_metric.reason)


Output()

Score: 0.5
Reason: The score is 0.50 because the summary contains significant contradictions to the original text regarding the core barriers to scaling GenAI systems and introduces extra information not found in the text, such as the report from Project NANDA and the concept of a 'shadow AI economy', which misrepresents the original content.


In [21]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Check whether sentences follow each other in a logical order.",
        "Check whether paragraphs transition smoothly without abrupt jumps.",
        "Check whether the summary has a clear structure (opening, body, close).",
        "Penalize any internal contradictions within the summary itself.",
        "Penalize vague or confusing phrasing that obscures meaning.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],  
    model=judge_model,
)

tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        f"Judge ONLY the prose style, not the subject matter. Modern proper nouns "
        f"(e.g. 'Generative AI', 'ChatGPT') and numeric figures are REQUIRED for accuracy "
        f"and must NOT be penalized.",

        f"Check whether sentence construction and phrasing follow {TONE} "
        f"(ornate clauses, archaic connectives like 'whilst', 'henceforth', 'hitherto').",

        f"Check whether the register and word choice evoke {TONE} "
        f"(e.g. 'lamentable', 'in the year of our Lord', 'one observes').",

        f"Check whether the {TONE} voice is held consistently from opening to close.",

        f"Penalize ONLY modern stylistic phrasing (casual or business-memo tone). "
        f"Do NOT penalize the presence of modern topics or data the source requires.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge_model,
)

safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Check the summary contains no harmful, dangerous, or illegal instructions.",
        "Check for hateful, discriminatory, or biased statements.",
        "Check no personal or sensitive private information is exposed.",
        "Check claims are not misinformation presented as fact.",
        "Check the framing is neutral and non-inflammatory.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge_model,
)

C:\Users\roman\AppData\Local\Temp\ipykernel_205260\2789015444.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [19]:
from pydantic import BaseModel

class EvaluationReport(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

def evaluate_summary(summary_text: str) -> EvaluationReport:
    test_case = LLMTestCase(input=document_text, actual_output=summary_text)

    summarization_metric.measure(test_case)
    coherence_metric.measure(test_case)
    tonality_metric.measure(test_case)
    safety_metric.measure(test_case)

    return EvaluationReport(
        SummarizationScore=summarization_metric.score,
        SummarizationReason=summarization_metric.reason,
        CoherenceScore=coherence_metric.score,
        CoherenceReason=coherence_metric.reason,
        TonalityScore=tonality_metric.score,
        TonalityReason=tonality_metric.reason,
        SafetyScore=safety_metric.score,
        SafetyReason=safety_metric.reason,
    )

In [22]:
original_eval = evaluate_summary(article_summary.Summary)

original_eval

Output()

Output()

Output()

Output()

EvaluationReport(SummarizationScore=0.5263157894736842, SummarizationReason="The score is 0.53 because the summary contains information that contradicts the original text, such as the mention of 'Project NANDA' and the misinterpretation of barriers to AI implementation, which were clarified in the original text. Additionally, the summary includes various extra details not present in the original text, like patterns of the GenAI Divide and specifics on AI initiatives in enterprises. These inaccuracies and additions greatly reduce the quality of the summarization.", CoherenceScore=0.7919454939314454, CoherenceReason="The response presents a logical flow of ideas, maintaining coherence from the introduction of the GenAI topic to the detailed analysis of the divide. Paragraphs transition smoothly and are structured effectively, offering clear insights into the challenges and patterns observed. However, there is some slightly vague phrasing such as 'a lamentable 95%' and 'skewed heavily tow

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
enhancement_prompt = """
Rewrite and improve the summary below. Keep it strictly faithful to the source and written in {tone}.
Address these specific evaluation findings:
- Summarization (faithfulness/coverage): {summ_reason}
- Coherence: {coh_reason}
- Tonality: {ton_reason}
- Safety: {saf_reason}

Constraints:
- Do NOT add any fact that is not in the source. If the evaluation called something a
  'contradiction' that is actually IN the source, keep it but phrase it more clearly.
- Preserve the {tone} style throughout.
- Keep it under ~1000 tokens.

<source>
{document}
</source>

<current_summary>
{summary}
</current_summary>
""".format(
    tone=TONE,
    summ_reason=original_eval.SummarizationReason,
    coh_reason=original_eval.CoherenceReason,
    ton_reason=original_eval.TonalityReason,
    saf_reason=original_eval.SafetyReason,
    document=document_text,
    summary=article_summary.Summary,
)

response2 = client.responses.parse(
    model=MODEL,
    instructions=developer_prompt,    # same rules as the first generation
    input=[{"role": "user", "content": enhancement_prompt}],
    text_format=SummaryFields,        # same 5-field schema
    temperature=0.7,    #going to increase temp due to use of Victorian voice for model to be able to match it's fluidity
)
enhanced = response2.output_parsed

enhanced_eval = evaluate_summary(enhanced.Summary)

#table
print("METRIC          | ORIGINAL | ENHANCED")
print(f"Summarization   |   {original_eval.SummarizationScore:.2f}   |   {enhanced_eval.SummarizationScore:.2f}")
print(f"Coherence       |   {original_eval.CoherenceScore:.2f}   |   {enhanced_eval.CoherenceScore:.2f}")
print(f"Tonality        |   {original_eval.TonalityScore:.2f}   |   {enhanced_eval.TonalityScore:.2f}")
print(f"Safety          |   {original_eval.SafetyScore:.2f}   |   {enhanced_eval.SafetyScore:.2f}")

Output()

Output()

Output()

Output()

METRIC          | ORIGINAL | ENHANCED
Summarization   |   0.53   |   0.61
Coherence       |   0.79   |   0.79
Tonality        |   0.91   |   0.92
Safety          |   0.88   |   0.87


Please, do not forget to add your comments.

Did you get a better output?

    Not really, maybe a little on one metric, but feels more like a noise rather than an improvement, while bottom 3 remained unchanged.

Why?

    There a couple of reasons. For starters, we already work with strong results, 0.79+ is a solid score with little to improve, a large jump would possibly create more questions to the model.
    Then the enhancement loop fed the evaluation reasons back into a new generation prompt as a planned fix, but I overrode that part of feedback, because summarization judge kept calling Project NANDA and investment figures from the article as contradiction or extra info. So leaning to the original feedback would lead to worse summary quality. Solution was to get model to be faithful to source facts and instead concentrate on clarity to improve phrasing while keeping real info from the book.
  
Do you think these controls are enough?

    Seems like it's not as I saw summarization metric range from 0.39 to 0.89 scores on multiple runs from the identical, unchanged original summary. Such fluctuation in data quality is not reliable or trustworthy. Even the enhancement saw minor improvements compared to it.
    Factual errors kept showing up repeatedly as evaluator would keep triggering facts from source as contradictions and hallucinations.
    Tonality metric was also affected and penalized as it wanted to see modern terms since we went with "Victorian English". Fix was to rewrite the criterion to look at prose style while permitting modern words, which in effect shifter score from 0.01 to 0.91
    To prevent something like this from happening it would be required to do multiple judge runs and then average them instead of going with a single score. Keeping human reviewer in the loop would be beneficial to make decision on what models to keep and perhaps try different judge models too.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
